# Memory in LLM Apps (Basic)

| Concept | Purpose |
| --- | --- |
| **Chat History** | Store current conversation |
| **Context Window** | Model's maximum available context |
| **Windowed Memory** | Keep recent messages |
| **Summarization** | Compress older messages |
| **Long-Term Memory** | Persist important information across conversations |

LLM has **no memory by itself**. Every time, we send it the past messages again. All 5 concepts are just different ways of deciding **what to send**.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## 1. Chat History

**Short note:** A list of all messages in the current conversation. We send the full list to the LLM every time, so it "remembers".

**Problem:** list keeps growing, cost goes up, and it can cross the limit.

In [ ]:
history = []

def chat(user_msg):
    history.append({"role": "user", "content": user_msg})
    res = llm.invoke(history)
    history.append({"role": "ai", "content": res.content})
    return res.content

print(chat("Hello, my name is Muktananda"))
print(chat("What is my name?"))
print(len(history), "messages stored")

## 2. Context Window

**Short note:** The maximum amount of text (measured in **tokens**) the model can read at once. Includes your history + new question + the answer.

- Token ≈ small piece of a word (1 token ≈ 4 characters in English)
- If history is bigger than the window → error, or old messages get cut

So we can't keep adding messages forever. That's why we need Windowed Memory and Summarization.

In [ ]:
# check how many tokens our history is using
text = " ".join(m["content"] for m in history)
print("Tokens used:", llm.get_num_tokens(text))
print("Characters :", len(text))

## 3. Windowed Memory

**Short note:** Send only the **last N messages**. Simple and cheap.

**Problem:** anything older than N is forgotten (like the user's name from message 1).

In [ ]:
history = []
WINDOW = 4   # keep last 4 messages only

def chat_windowed(user_msg):
    history.append({"role": "user", "content": user_msg})

    recent = history[-WINDOW:]          # only the last 4
    res = llm.invoke(recent)

    history.append({"role": "ai", "content": res.content})
    return res.content

chat_windowed("Hi, my name is Muktananda")
chat_windowed("I like Java")
chat_windowed("I also like Python")
chat_windowed("Tell me one fun fact")
print(chat_windowed("What is my name?"))   # name is outside the window, so it may forget

## 4. Summarization

**Short note:** Instead of throwing old messages away, **compress them into a short summary**. Send: `summary + recent messages`.

Keeps the important points, uses fewer tokens.

In [ ]:
history = []
summary = ""
KEEP = 4   # keep last 4 messages as they are

def summarize(old_messages, old_summary):
    text = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)
    prompt = f"Existing summary: {old_summary}\n\nNew messages:\n{text}\n\nWrite a short updated summary."
    return llm.invoke(prompt).content

def chat_summary(user_msg):
    global summary, history
    history.append({"role": "user", "content": user_msg})

    # if history is getting long, compress the old part
    if len(history) > KEEP + 2:
        old, history = history[:-KEEP], history[-KEEP:]
        summary = summarize(old, summary)

    messages = []
    if summary:
        messages.append({"role": "system", "content": f"Summary of earlier chat: {summary}"})
    messages += history

    res = llm.invoke(messages)
    history.append({"role": "ai", "content": res.content})
    return res.content

for m in ["Hi, my name is Muktananda", "I like Java", "I use Spring Boot", "Tell me a fun fact", "Another one", "What is my name?"]:
    print("User:", m)
    print("AI  :", chat_summary(m), "\n")

print("SUMMARY:", summary)

## 5. Long-Term Memory

**Short note:** Save important facts **outside the chat** (file / database), so a **new conversation** can still use them.

Flow: `save fact → new chat → load facts → give to LLM`

Below is the simplest version: a JSON file. (Real apps use a database or vector DB and pick only the *relevant* facts.)

In [ ]:
import json, os

FILE = "memory.json"

def load_memory():
    return json.load(open(FILE)) if os.path.exists(FILE) else []

def save_memory(fact):
    facts = load_memory()
    if fact not in facts:
        facts.append(fact)
        json.dump(facts, open(FILE, "w"), indent=2)

# --- Conversation 1 (today) ---
save_memory("User's favorite language is Java")
save_memory("User uses Spring Boot")

In [ ]:
# --- Conversation 2 (new chat, history is empty) ---
def chat_longterm(user_msg):
    facts = load_memory()
    system = "You know these facts about the user:\n" + "\n".join(f"- {f}" for f in facts)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    return llm.invoke(messages).content

print(chat_longterm("What programming language do I like?"))

## Quick Summary

| Concept | One line | Code idea |
| --- | --- | --- |
| Chat History | all messages of this chat | `history.append(...)` |
| Context Window | model's limit | `llm.get_num_tokens(text)` |
| Windowed Memory | last N messages | `history[-4:]` |
| Summarization | old messages → short summary | `summary + recent` |
| Long-Term Memory | facts saved in a file/DB | `save_memory()` / `load_memory()` |